<h1> Supplier Advanced Synthetic Boosted Data </h1>

In [26]:
import os
import random
import ast
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.auto import tqdm  # Works in both local and Kaggle

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch: 2.9.0+cu126
CUDA: True
GPU: Tesla P100-PCIE-16GB


In [27]:
# Aircraft dataset path
DATA_DIR = "/kaggle/input/datasets/gururockzz/ss-supplier-v2/suppliers_augmented_v2"
# DATA_DIR = "segregated_datasets/aircraft"  # Relative path option

OUTPUT_DIR = "suppliers_augmented_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training settings
IMG_H, IMG_W = 48, 128
BATCH_SIZE = 64
EPOCHS = 150
LR = 1e-4
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Load classes from data.yaml

def load_class_names(data_dir):
    data_yaml = Path(data_dir) / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(f"data.yaml not found at: {data_yaml}")

    names = []
    for raw in data_yaml.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if line.startswith("names:"):
            parsed = ast.literal_eval(line.split(":", 1)[1].strip())
            if isinstance(parsed, list):
                names = [str(x) for x in parsed]
            break

    if not names:
        raise ValueError(f"Could not parse class names from: {data_yaml}")
    return names

CLASS_NAMES = load_class_names(DATA_DIR)
NUM_TOKENS = len(CLASS_NAMES)
ID2CHAR = {i: name for i, name in enumerate(CLASS_NAMES)}
CHAR2ID = {name: i for i, name in ID2CHAR.items()}

# CTC setup
BLANK = NUM_TOKENS
NUM_CLASSES = NUM_TOKENS + 1

# Optional focus tokens for weighted CTC (leave empty to disable boosting)
FOCUS_TOKENS = []  # Example: ["I", "O"]
FOCUS_TOKEN_IDS = {CHAR2ID[t] for t in FOCUS_TOKENS if t in CHAR2ID}

print(f"Classes ({NUM_TOKENS}): {CLASS_NAMES}")
print(f"Focus tokens: {[ID2CHAR[i] for i in sorted(FOCUS_TOKEN_IDS)]}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Classes (26): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Focus tokens: []


## Load & Verify Data

In [28]:
def ids_to_text(token_ids):
    # Keep multi-character classes readable (e.g., nan -> <nan>)
    out = []
    for tid in token_ids:
        token = ID2CHAR[int(tid)]
        out.append(token if len(token) == 1 else f"<{token}>")
    return ''.join(out)

def parse_yolo_to_ids(label_path):
    """Convert YOLO labels to ordered class-id sequence (left-to-right)."""
    if not os.path.exists(label_path):
        return []

    boxes = []
    with open(label_path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                cls = int(parts[0])
                x = float(parts[1])
                if 0 <= cls < NUM_TOKENS:
                    boxes.append((cls, x))

    boxes.sort(key=lambda b: b[1])
    return [cls for cls, _ in boxes]

def load_data(data_dir):
    data = {}
    for split in ["train", "valid", "test"]:
        img_dir = Path(data_dir) / split / "images"
        lbl_dir = Path(data_dir) / split / "labels"

        if not img_dir.exists():
            print(f"Warning: {split} not found")
            continue

        samples = []
        image_files = sorted([p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

        for img_path in image_files:
            lbl_path = lbl_dir / f"{img_path.stem}.txt"
            token_ids = parse_yolo_to_ids(str(lbl_path))
            if token_ids:
                samples.append({
                    "img": str(img_path),
                    "ids": token_ids,
                    "text": ids_to_text(token_ids)
                })

        data[split] = samples
        print(f"{split}: {len(samples)} samples")

    return data

data = load_data(DATA_DIR)

# Analyze token distribution
token_counts = Counter(t for s in data["train"] for t in s["ids"])
print("\nToken distribution in training:")
for tid in range(NUM_TOKENS):
    print(f"  {ID2CHAR[tid]} ({tid}): {token_counts.get(tid, 0)}")

train: 33746 samples
valid: 3500 samples
test: 3462 samples

Token distribution in training:
  A (0): 6314
  B (1): 6267
  C (2): 6325
  D (3): 6247
  E (4): 6280
  F (5): 6277
  G (6): 6230
  H (7): 6236
  I (8): 6325
  J (9): 6241
  K (10): 6210
  L (11): 6335
  M (12): 6248
  N (13): 6258
  O (14): 6364
  P (15): 6290
  Q (16): 6226
  R (17): 6264
  S (18): 6321
  T (19): 6250
  U (20): 6221
  V (21): 6222
  W (22): 6241
  X (23): 0
  Y (24): 0
  Z (25): 6243


## Augmentation

In [29]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

# ------------------------------
# STANDARD (Safe) AUGMENTATION
# ------------------------------
train_aug = A.Compose([
    A.Resize(IMG_H, IMG_W),

    # Very mild geometric variation
    A.ShiftScaleRotate(
        shift_limit=0.03,
        scale_limit=0.05,
        rotate_limit=3,
        border_mode=cv2.BORDER_REPLICATE,
        p=0.5
    ),

    # Light noise
    A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),

    # Light blur (realistic camera blur)
    A.GaussianBlur(blur_limit=3, p=0.2),

    # Mild brightness/contrast change
    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.3
    ),

    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2(),
])


# ------------------------------
# STRONG (BUT SAFE) AUGMENTATION
# For confusing characters (O/0, I/1 etc.)
# ------------------------------
strong_aug = A.Compose([
    A.Resize(IMG_H, IMG_W),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.08,
        rotate_limit=5,
        border_mode=cv2.BORDER_REPLICATE,
        p=0.6
    ),

    # Slight additional noise
    A.GaussNoise(var_limit=(10.0, 30.0), p=0.4),

    A.GaussianBlur(blur_limit=3, p=0.3),

    A.RandomBrightnessContrast(
        brightness_limit=0.2,
        contrast_limit=0.2,
        p=0.4
    ),

    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2(),
])


# ------------------------------
# VALIDATION (NO AUGMENTATION)
# ------------------------------
val_aug = A.Compose([
    A.Resize(IMG_H, IMG_W),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2(),
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_55/2511727941.py:21: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),
/tmp/ipykernel_55/2511727941.py:54: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 30.0), p=0.4),


## Dataset with Oversampling for Confused Digits

In [30]:
class OCRDataset(Dataset):
    """Dataset with optional oversampling for samples containing focus tokens."""

    def __init__(self, samples, transform, strong_transform=None,
                 confused_token_ids=None, oversample_factor=2):
        self.samples = samples
        self.transform = transform
        self.strong_transform = strong_transform or transform
        self.confused_token_ids = confused_token_ids or set()
        self.oversample_factor = oversample_factor

        # Build oversampled index list
        self.indices = []
        self.is_focus = []

        for i, s in enumerate(samples):
            has_focus = any(t in self.confused_token_ids for t in s["ids"])

            if has_focus and oversample_factor > 1:
                self.indices.extend([i] * oversample_factor)
                self.is_focus.extend([True] * oversample_factor)
            else:
                self.indices.append(i)
                self.is_focus.append(has_focus)

        if self.confused_token_ids:
            print(f"Original: {len(samples)}, After oversampling: {len(self.indices)}")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        s = self.samples[real_idx]

        img = cv2.imread(s["img"], cv2.IMREAD_GRAYSCALE)
        img = np.stack([img] * 3, axis=-1)

        # Use stronger augmentation for focus-token samples
        if self.is_focus[idx]:
            img = self.strong_transform(image=img)["image"]
        else:
            img = self.transform(image=img)["image"]

        img = img[0:1]
        label = s["ids"]

        return img, torch.tensor(label), len(label)

def collate(batch):
    imgs, labels, lengths = zip(*batch)
    imgs = torch.stack(imgs)
    max_len = max(lengths)
    padded = torch.full((len(labels), max_len), BLANK, dtype=torch.long)
    for i, (lbl, ln) in enumerate(zip(labels, lengths)):
        padded[i, :ln] = lbl
    return imgs, padded, torch.tensor(lengths)

# Create datasets with optional oversampling
train_ds = OCRDataset(
    data["train"], train_aug, strong_aug,
    confused_token_ids=FOCUS_TOKEN_IDS, oversample_factor=4
)
val_ds = OCRDataset(data["valid"], val_aug)
test_ds = OCRDataset(data["test"], val_aug)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, collate_fn=collate, pin_memory=True,num_workers=3)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, collate_fn=collate, pin_memory=True)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, collate_fn=collate, pin_memory=True)

## VGG-BiLSTM Model

In [31]:
class VGG_OCR(nn.Module):
    """VGG-style CNN + BiLSTM for OCR."""

    def __init__(self, num_classes=NUM_CLASSES, hidden=256, num_layers=3):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),

            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),

            nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, 512, kernel_size=(3,1)),
            nn.BatchNorm2d(512),
            nn.ReLU(True)
        )

        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.fc = nn.Linear(hidden * 2, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        conv = self.cnn(x)
        conv = conv.squeeze(2).permute(0, 2, 1)
        rnn_out, _ = self.rnn(conv)
        out = self.fc(rnn_out)
        return F.log_softmax(out, dim=2)

model = VGG_OCR(NUM_CLASSES, hidden=256, num_layers=3).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 12,581,595


In [32]:
#OLD
# class VGG_OCR(nn.Module):
#     """VGG-style CNN + BiLSTM for OCR."""

#     def __init__(self, num_classes=NUM_CLASSES, hidden=256, num_layers=3):
#         super().__init__()

#         self.cnn = nn.Sequential(
#             nn.Conv2d(1, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
#             nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
#             nn.MaxPool2d(2, 2),

#             nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
#             nn.Conv2d(128, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
#             nn.MaxPool2d(2, 2),

#             nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
#             nn.Conv2d(256, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
#             nn.MaxPool2d((2, 1), (2, 1)),

#             nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
#             nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
#             nn.MaxPool2d((2, 1), (2, 1)),

#             nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
#             nn.AdaptiveAvgPool2d((1, None))
#         )

#         self.rnn = nn.LSTM(
#             input_size=512,
#             hidden_size=hidden,
#             num_layers=num_layers,
#             batch_first=True,
#             bidirectional=True,
#             dropout=0.3
#         )

#         self.fc = nn.Linear(hidden * 2, num_classes)
#         self._init_weights()

#     def _init_weights(self):
#         for m in self.modules():
#             if isinstance(m, nn.Conv2d):
#                 nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
#             elif isinstance(m, nn.BatchNorm2d):
#                 nn.init.constant_(m.weight, 1)
#                 nn.init.constant_(m.bias, 0)
#             elif isinstance(m, nn.Linear):
#                 nn.init.xavier_uniform_(m.weight)
#                 nn.init.constant_(m.bias, 0)

#     def forward(self, x):
#         conv = self.cnn(x)
#         conv = conv.squeeze(2).permute(0, 2, 1)
#         rnn_out, _ = self.rnn(conv)
#         out = self.fc(rnn_out)
#         return F.log_softmax(out, dim=2)

# model = VGG_OCR(NUM_CLASSES, hidden=256, num_layers=2).to(DEVICE)
# print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Decoding and Metrics

In [33]:
def greedy_decode(output, blank=BLANK):
    """Greedy CTC decoding -> list of token-id sequences."""
    preds = []
    for seq in output:
        indices = seq.argmax(dim=1).cpu().numpy().tolist()
        result = []
        prev = -1
        for idx in indices:
            if idx != prev and idx != blank:
                result.append(int(idx))
            prev = idx
        preds.append(result)
    return preds

def compute_cer(pred_tokens, target_tokens):
    """Token Error Rate using edit distance over token ids."""
    if len(target_tokens) == 0:
        return 1.0 if len(pred_tokens) > 0 else 0.0

    m, n = len(pred_tokens), len(target_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_tokens[i - 1] == target_tokens[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    return dp[m][n] / len(target_tokens)

def compute_metrics(preds, labels, lengths):
    """Compute exact-sequence accuracy and token error rate."""
    correct = 0
    total_cer = 0.0

    for pred, label, ln in zip(preds, labels, lengths):
        target = [int(l.item()) for l in label[:ln]]

        if pred == target:
            correct += 1

        total_cer += compute_cer(pred, target)

    acc = correct / len(preds)
    avg_cer = total_cer / len(preds)
    return acc, avg_cer

## Confusion Analysis

In [34]:
def analyze_confusions(model, loader, device):
    """Analyze token-level confusions and return confused token ids."""
    model.eval()
    confusion = np.zeros((NUM_TOKENS, NUM_TOKENS), dtype=int)

    with torch.no_grad():
        for imgs, labels, lens in loader:
            imgs = imgs.to(device)
            out = model(imgs)
            preds = greedy_decode(out)

            for pred, label, ln in zip(preds, labels, lens):
                target = [int(l.item()) for l in label[:ln]]
                for j in range(min(len(pred), len(target))):
                    confusion[target[j]][pred[j]] += 1

    print("\n" + "=" * 60)
    print("CONFUSION ANALYSIS")
    print("=" * 60)

    # Find confused pairs
    confused_token_ids = set()
    confusions = []
    for i in range(NUM_TOKENS):
        for j in range(NUM_TOKENS):
            if i != j and confusion[i][j] > 0:
                confusions.append((i, j, confusion[i][j]))
                if confusion[i][j] >= 5:  # Threshold
                    confused_token_ids.add(i)
                    confused_token_ids.add(j)

    confusions.sort(key=lambda x: -x[2])
    print("\nTop confused pairs (True -> Pred : Count):")
    for true, pred, count in confusions[:20]:
        print(f"  {ID2CHAR[true]} -> {ID2CHAR[pred]} : {count}")

    print("\nPer-token accuracy:")
    for i in range(NUM_TOKENS):
        total = confusion[i].sum()
        if total > 0:
            acc = confusion[i][i] / total * 100
            print(f"  {ID2CHAR[i]}: {acc:.1f}%")

    return confused_token_ids, confusion

## Weighted CTC Loss

In [35]:
class WeightedCTCLoss(nn.Module):
    """CTC Loss with higher weight for samples containing selected focus tokens."""

    def __init__(self, blank=BLANK, confused_token_ids=None, weight_boost=1.5):
        super().__init__()
        self.ctc = nn.CTCLoss(blank=blank, zero_infinity=True, reduction="none")
        self.confused_token_ids = confused_token_ids or set()
        self.weight_boost = weight_boost

    def forward(self, log_probs, targets, input_lengths, target_lengths):
        # Get per-sample losses
        losses = self.ctc(log_probs, targets, input_lengths, target_lengths)

        # Apply optional weights
        weights = torch.ones_like(losses)

        if self.confused_token_ids:
            for i in range(targets.size(0)):
                target_len = target_lengths[i].item()
                target = targets[i, :target_len]

                has_focus = any(int(d.item()) in self.confused_token_ids for d in target)
                if has_focus:
                    weights[i] = self.weight_boost

        weighted_loss = (losses * weights).mean()
        return weighted_loss

## Training Functions

## Training Setup

In [36]:
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels, all_lens = [], [], []
    
    for imgs, labels, lens in tqdm(loader, desc="Train"):
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast("cuda"):
            out = model(imgs)
            out_t = out.permute(1, 0, 2)
            input_lens = torch.full((out_t.size(1),), out_t.size(0), dtype=torch.long)
            loss = criterion(out_t, labels, input_lens, lens)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        with torch.no_grad():
            preds = greedy_decode(out)
            all_preds.extend(preds)
            all_labels.extend(labels.cpu())
            all_lens.extend(lens.cpu())
    
    acc, cer = compute_metrics(all_preds, all_labels, all_lens)
    return total_loss / len(loader), acc, cer

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_lens = [], [], []
    
    for imgs, labels, lens in tqdm(loader, desc="Eval"):
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        out = model(imgs)
        out_t = out.permute(1, 0, 2)
        input_lens = torch.full((out_t.size(1),), out_t.size(0), dtype=torch.long)
        loss = criterion(out_t, labels, input_lens, lens)
        
        total_loss += loss.item()
        preds = greedy_decode(out)
        all_preds.extend(preds)
        all_labels.extend(labels.cpu())
        all_lens.extend(lens.cpu())
    
    acc, cer = compute_metrics(all_preds, all_labels, all_lens)
    return total_loss / len(loader), acc, cer

In [37]:
# Weighted CTC (boosting only applies if FOCUS_TOKEN_IDS is not empty)
criterion = WeightedCTCLoss(blank=BLANK, confused_token_ids=FOCUS_TOKEN_IDS, weight_boost=1.3)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = GradScaler()


/tmp/ipykernel_55/138623911.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


## 🚀 Training Loop

In [ ]:
best_acc = 0
patience = 5
no_improve = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

focus_names = [ID2CHAR[i] for i in sorted(FOCUS_TOKEN_IDS)]
print("=" * 60)
print(f"Training with focus tokens: {focus_names if focus_names else 'None'}")
print(f"Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | Device: {DEVICE}")
print("=" * 60)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_acc, train_cer = train_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    val_loss, val_acc, val_cer = evaluate(model, val_loader, criterion, DEVICE)

    scheduler.step(val_acc)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train: Loss={train_loss:.4f} Acc={train_acc * 100:.2f}% TER={train_cer * 100:.1f}%")
    print(f"Valid: Loss={val_loss:.4f} Acc={val_acc * 100:.2f}% TER={val_cer * 100:.1f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        no_improve = 0
        torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model.pth")
        print(f"Best model saved: {best_acc * 100:.2f}%")
    else:
        no_improve += 1
        if no_improve >= patience:
            print("\nEarly stopping")
            break

    # Analyze confusions every 20 epochs
    if (epoch + 1) % 20 == 0:
        analyze_confusions(model, val_loader, DEVICE)

print(f"\n{'=' * 60}")
print(f"Training Complete! Best: {best_acc * 100:.2f}%")
print(f"{'=' * 60}")

Training with focus tokens: None
Epochs: 150 | Batch: 64 | Device: cuda

Epoch 1/150


Train:   0%|          | 0/528 [00:00<?, ?it/s]

## Test Evaluation & Final Confusion Analysis

In [ ]:
# Load best model
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model.pth"))

# Evaluate on test set
test_loss, test_acc, test_cer = evaluate(model, test_loader, criterion, DEVICE)

print(f"\n{'=' * 60}")
print("TEST RESULTS")
print(f"{'=' * 60}")
print(f"Accuracy: {test_acc * 100:.2f}%")
print(f"TER: {test_cer * 100:.2f}%")

# Final confusion analysis
confused_token_ids, confusion_matrix = analyze_confusions(model, test_loader, DEVICE)

## Visualize Results

In [ ]:
model.eval()
fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for i, ax in enumerate(axes.flat):
    if i >= len(data["test"]):
        break
    s = data["test"][i]
    img = cv2.imread(s["img"], cv2.IMREAD_GRAYSCALE)
    tensor = val_aug(image=np.stack([img] * 3, -1))["image"][0:1].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(tensor)
        pred_ids = greedy_decode(out)[0]

    pred_text = ids_to_text(pred_ids)
    color = "green" if pred_ids == s["ids"] else "red"
    ax.imshow(img, cmap="gray")
    ax.set_title(f"GT:{s['text']}\nPred:{pred_text}", color=color, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/predictions.png")
plt.show()

In [ ]:
# Plot confusion matrix
labels = [ID2CHAR[i] for i in range(NUM_TOKENS)]

plt.figure(figsize=(16, 14))
plt.imshow(confusion_matrix, cmap="Blues")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Aircraft Token Confusion Matrix")
plt.xticks(range(NUM_TOKENS), labels, rotation=90)
plt.yticks(range(NUM_TOKENS), labels)

# Add text annotations
max_val = confusion_matrix.max() if confusion_matrix.size > 0 else 0
for i in range(NUM_TOKENS):
    for j in range(NUM_TOKENS):
        val = confusion_matrix[i][j]
        if val > 0:
            color = "white" if max_val > 0 and val > max_val / 2 else "black"
            plt.text(j, i, str(val), ha="center", va="center", color=color)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png")
plt.show()